In [ ]:
import torch
import torch.nn.functional as F
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import sys
sys.path.append('')

from model.encoder_dino_0927 import EncoderDino
import os
import cv2

def visualize_feature_similarity(features: torch.Tensor, row: int, col: int):
    """
    计算特征矩阵中所有特征与指定锚点(row, col)特征的余弦相似度，
    并使用 Matplotlib 进行可视化。

    参数:
    features (torch.Tensor): 输入的特征矩阵，形状为 (H, W, D)。
    row (int): 锚点特征的行号。
    col (int): 锚点特征的列号。
    """
    if features.dim() != 3:
        print(f"错误: 输入的特征矩阵应为3维 (H, W, D)，但得到的是 {features.dim()} 维。")
        return
    
    H, W, D = features.shape
    
    # 2. 检查锚点坐标是否越界
    if not (0 <= row < H and 0 <= col < W):
        print(f"错误: 锚点坐标 ({row}, {col}) 超出了矩阵范围 ({H}, {W})。")
        return

    print(f"正在计算与锚点 ({row}, {col}) 的相似度...")
    anchor_feat = features[row, col, :]
    features_norm = F.normalize(features, p=2, dim=2)
    anchor_feat_norm = F.normalize(anchor_feat, p=2, dim=0)
    similarity_map = torch.einsum('hwd,d->hw', features_norm, anchor_feat_norm)
    similarity_map_np = similarity_map.cpu().detach().numpy()
    fig, ax = plt.subplots()
    ax.imshow(similarity_map_np, cmap='plasma', vmin=-1, vmax=1)
    ax.plot(col, row, 'r+', markersize=15, markeredgewidth=3)
    ax.axis('off')
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
    print("可视化完成。")
    plt.show()

In [ ]:
encoder_path = ''
img_path = ''

In [ ]:
encoder = EncoderDino(
            os.path.join('weights/dinov3_vitl16_pretrain_sat493m-eadcf0ff.pth'),
            upsample_times=0,
            use_adapter=True,
            use_conf = True
        )
encoder.load_adapter(os.path.join(encoder_path,'adapter.pth'))
encoder.cuda()
encoder.eval()

transform = transforms.Compose([
                    transforms.ToTensor(),
                    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225)) 
                    ])

img = cv2.imread(img_path)
img_tensor = transform(img)[None].cuda()
feat,_ = encoder(img_tensor)
feat = feat[0].permute(1,2,0)

In [ ]:
visualize_feature_similarity(feat,32,32)